# Bivariate and multivariate analysis

Run each cell in order with **Shift + Enter**. Each step matches a section of the Note.
Every chart is drawn with Plotly: hover over a point, bar or cell to read its exact values; drag to zoom; double-click to reset.

Data (all in `data/`): `tips.csv` (restaurant bills), `titanic_train.csv` (passengers), `flights.csv` (monthly air passengers), `iris.csv` (flowers).

## 1. Load the four datasets

In [1]:
import pandas as pd
import plotly.express as px
import numpy as np
from scipy.stats import gaussian_kde
from scipy.cluster.hierarchy import linkage, leaves_list

# the same colours as the Note's figures
BLUE, ORANGE, GREEN, RED = "#4C78A8", "#F58518", "#54A24B", "#E45756"
pd.__version__

'3.0.6'

In [2]:
# tips, flights and iris ship with seaborn (sns.load_dataset); we keep local copies
tips = pd.read_csv("data/tips.csv")
titanic = pd.read_csv("data/titanic_train.csv")
flights = pd.read_csv("data/flights.csv")
iris = pd.read_csv("data/iris.csv")
tips.head()

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4


In [3]:
flights.head()

,year,month,passengers
0,1949,Jan,112
1,1949,Feb,118
2,1949,Mar,132
3,1949,Apr,129
4,1949,May,121


In [4]:
iris.head()

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


## 2. Scatter plot (numerical + numerical)

In [5]:
# one dot per bill: total bill on x, tip on y
px.scatter(tips, x="total_bill", y="tip")

In [6]:
# tip as a share of the bill: median and middle half
share = tips["tip"] / tips["total_bill"]
share.quantile([0.25, 0.5, 0.75]).round(3)

0.25    0.129
0.50    0.155
0.75    0.191
dtype: float64

In [7]:
# multivariate: colour = sex, marker = smoker, dot size = party size (five columns in one plot)
px.scatter(tips, x="total_bill", y="tip", color="sex", symbol="smoker", size="size",
           color_discrete_map={"Male": BLUE, "Female": ORANGE})

## 3. Bar plot (numerical + categorical)

In [8]:
# the bar plot shows the mean age of each class; this is the number behind each bar
titanic.groupby("Pclass")["Age"].mean().round(1)

Pclass
1    38.2
2    29.9
3    25.1
Name: Age, dtype: float64

In [9]:
# mean age per class and sex (hue = sex), as a table and then as bars
mean_age = titanic.groupby(["Pclass", "Sex"], as_index=False)["Age"].mean()
fig = px.bar(mean_age, x="Pclass", y="Age", color="Sex", barmode="group",
             color_discrete_map={"male": BLUE, "female": ORANGE})
fig.update_xaxes(type="category")
fig

## 4. Box plot (numerical + categorical)

In [10]:
# age by sex, one box per outcome (hue = Survived)
titanic["Outcome"] = titanic["Survived"].map({0: "died", 1: "survived"})
px.box(titanic, x="Sex", y="Age", color="Outcome",
       color_discrete_map={"died": RED, "survived": GREEN})

## 5. KDE plot (numerical + categorical)

In [11]:
# split the ages by outcome, dropping missing ages
died = titanic.loc[titanic["Survived"] == 0, "Age"].dropna()
survived = titanic.loc[titanic["Survived"] == 1, "Age"].dropna()

# one smooth density curve per group (the old sns.distplot(..., hist=False))
# gaussian_kde builds the curve; we read it at 200 ages from 0 to 80
ages = np.linspace(0, 80, 200)
curves = pd.DataFrame({"Age": ages,
                       "died": gaussian_kde(died)(ages),
                       "survived": gaussian_kde(survived)(ages)})
px.line(curves, x="Age", y=["died", "survived"], labels={"value": "Density", "variable": ""},
        color_discrete_map={"died": RED, "survived": GREEN})

## 6. Heatmap of a crosstab (categorical + categorical)

In [12]:
# count passengers for every (class, outcome) pair
ct = pd.crosstab(titanic["Pclass"], titanic["Survived"])
ct

Survived,0,1
Pclass,,
1,80,136
2,97,87
3,372,119


In [13]:
# the same table, coloured by size
px.imshow(ct, text_auto=True, color_continuous_scale="Blues", aspect="auto")

In [14]:
# survival rate (%) per class: the mean of a 0/1 column is the share of 1s
(titanic.groupby("Pclass")["Survived"].mean() * 100).round(1)

Pclass
1    63.0
2    47.3
3    24.2
Name: Survived, dtype: float64

In [15]:
# the same for sex and port of boarding
for col in ["Sex", "Embarked"]:
    print((titanic.groupby(col)["Survived"].mean() * 100).round(1), "\n")

Sex
female    74.2
male      18.9
Name: Survived, dtype: float64 

Embarked
C    55.4
Q    39.0
S    33.7
Name: Survived, dtype: float64 



In [16]:
# a series plots straight to a bar chart
rate = titanic.groupby("Embarked")["Survived"].mean() * 100
px.bar(rate, labels={"value": "Survived (%)"})

## 7. Clustermap (categorical + categorical)

In [17]:
def clustered(table):
    """Reorder rows and columns so similar ones sit together (what a clustermap does)."""
    rows = leaves_list(linkage(table.values, method="average"))
    cols = leaves_list(linkage(table.values.T, method="average"))
    return table.iloc[rows, cols]

# crosstab of parents/children aboard vs survival, rows grouped by similarity
parch = pd.crosstab(titanic["Parch"], titanic["Survived"])
px.imshow(clustered(parch), text_auto=True, color_continuous_scale="Blues", aspect="auto")

In [18]:
# the same idea for siblings/spouses aboard
sibsp = pd.crosstab(titanic["SibSp"], titanic["Survived"])
px.imshow(clustered(sibsp), text_auto=True, color_continuous_scale="Blues", aspect="auto")

## 8. Pair plot (many numerical columns)

In [19]:
# every pair of the four measurements, coloured by species
cols = ["sepal_length", "sepal_width", "petal_length", "petal_width"]
fig = px.scatter_matrix(iris, dimensions=cols, color="species",
                        color_discrete_sequence=[BLUE, ORANGE, GREEN], height=750)
fig.update_traces(diagonal_visible=False, marker_size=4)
fig

## 9. Line plot (numerical over time)

In [20]:
# total passengers per year: group by year, add up the 12 months
yearly = flights.groupby("year", as_index=False)["passengers"].sum()
yearly

,year,passengers
0,1949,1520
1,1950,1676
2,1951,2042
3,1952,2364
4,1953,2700
5,1954,2867
6,1955,3408
7,1956,3939
8,1957,4421
9,1958,4572


In [21]:
px.line(yearly, x="year", y="passengers", markers=True)

## 10. Pivot table, heatmap and clustermap of the flights

In [22]:
# month order is lost in a CSV (it sorts alphabetically), so we set it back
months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
          "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

# one row per month, one column per year, passengers in the cells
table = flights.pivot_table(values="passengers", index="month", columns="year")
table = table.reindex(months)
table

year,1949,1950,1951,1952,1953,1954,1955,1956,1957,1958,1959,1960
month,,,,,,,,,,,,
Jan,112.0,115.0,145.0,171.0,196.0,204.0,242.0,284.0,315.0,340.0,360.0,417.0
Feb,118.0,126.0,150.0,180.0,196.0,188.0,233.0,277.0,301.0,318.0,342.0,391.0
Mar,132.0,141.0,178.0,193.0,236.0,235.0,267.0,317.0,356.0,362.0,406.0,419.0
Apr,129.0,135.0,163.0,181.0,235.0,227.0,269.0,313.0,348.0,348.0,396.0,461.0
May,121.0,125.0,172.0,183.0,229.0,234.0,270.0,318.0,355.0,363.0,420.0,472.0
Jun,135.0,149.0,178.0,218.0,243.0,264.0,315.0,374.0,422.0,435.0,472.0,535.0
Jul,148.0,170.0,199.0,230.0,264.0,302.0,364.0,413.0,465.0,491.0,548.0,622.0
Aug,148.0,170.0,199.0,242.0,272.0,293.0,347.0,405.0,467.0,505.0,559.0,606.0
Sep,136.0,158.0,184.0,209.0,237.0,259.0,312.0,355.0,404.0,404.0,463.0,508.0


In [23]:
px.imshow(table, color_continuous_scale="Blues", aspect="auto")

In [24]:
# clustermap: similar months and similar years moved next to each other
px.imshow(clustered(table), text_auto=True, color_continuous_scale="Blues", aspect="auto", height=550)